# Airtel Enterprise Customer Churn — Prediction Model

## 1. Problem Definition

Binary classification: predict whether an enterprise customer will churn (`Churn` = 1) based on service usage, service quality, support experience, satisfaction, and competitor-pressure signals. The output is a **churn probability** per customer, which the business uses for risk tiering and retention prioritization — not just a hard yes/no label.

## 2. Target Variable

`Churn` — 0 = Active, 1 = Churned. Base rate in this dataset: ~21.4% (moderately imbalanced).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
sys.path.insert(0, '../src')

from feature_engineering import prepare_model_matrix, engineer_features, DROP_COLS, CATEGORICAL_COLS, TARGET

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

df = pd.read_csv('../data/airtel_enterprise_churn.csv')
print(df.shape)
df['Churn'].value_counts(normalize=True)

## 3. Feature Selection

Dropped before modeling:
- **Identifiers / free text**: `Customer_ID`, `Company_Name`, `Primary_Service`, `Competitor_Offer`
- **Target-leakage fields**: `Churn_Date`, `Churn_Reason`, `Churn_Category` — these only exist *because* a customer churned, so including them would let the model "cheat"
- **Redundant collinear features** (checked via correlation, see below): `Monthly_Bill` (r=0.999 with `Annual_Contract_Value`), `Monthly_Data_Usage_GB` (r=0.967 with `Bandwidth_Mbps`), `Number_of_Service_Tickets` (r=0.927 with `Number_of_Complaints`), `Support_Resolution_Hours` (r=0.928 with `Support_Response_Hours`)

Near-duplicate features don't add predictive signal but do destabilize a linear model's coefficients — a first pass at this model actually put `Monthly_Bill`/`Annual_Contract_Value` as the top two "drivers" of churn purely because of this collinearity, which was a misleading result. Dropping the redundant half of each pair fixed it and let the real service-quality drivers surface.

In [ ]:
corr_check = df[['Annual_Contract_Value','Monthly_Bill','Bandwidth_Mbps','Monthly_Data_Usage_GB',
                  'Number_of_Complaints','Number_of_Service_Tickets',
                  'Support_Response_Hours','Support_Resolution_Hours']].corr()
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(corr_check, annot=True, fmt='.3f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Near-duplicate feature pairs (checked for multicollinearity)')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing & Feature Engineering

`src/feature_engineering.py` adds several derived features on top of the raw columns:

- **CLV** — Customer Lifetime Value, `Annual_Contract_Value x expected remaining lifetime` (lifetime horizon shrinks with lower satisfaction)
- **Tickets_per_Tenure_Year** — support burden normalized by relationship length
- **Complaint_Satisfaction_Gap** — complaints weighted by (10 - satisfaction), flags customers complaining despite being nominally "satisfied"
- **Contract_Ending_Soon** — binary flag, contract renewal window (<=3 months)
- **Price_Pressure** — competitor price advantage, zeroed out unless a competitor was actually considered
- **Revenue_Weighted_Downtime** — downtime scaled by account value (a high-value account's downtime matters more)
- **Services_per_Size_Tier** — service breadth relative to what's typical for that company size

Categorical columns (`Company_Size`, `Industry`, `State`, `City`, `Contract_Type`, etc.) are one-hot encoded.

In [ ]:
X, y, encoder_categories = prepare_model_matrix(df)
X = X.astype(np.float64)
print(f"Feature matrix: {X.shape}")
X.head()

## 5. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean()*100:.2f}% | Test churn rate: {y_test.mean()*100:.2f}%")

## 6. Baseline Model

A simple "always predict majority class" baseline gives ~78.6% accuracy (since ~21.4% churn) purely by predicting everyone stays — which is exactly why accuracy alone is a poor metric here. Any real model needs to beat this on **recall and ROC-AUC**, not just accuracy.

In [ ]:
baseline_acc = 1 - y_test.mean()
print(f"Majority-class baseline accuracy: {baseline_acc*100:.2f}%")
print("This baseline has 0% recall on churners — completely useless for the actual business problem.")

## 7. Multiple ML Models

Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting. **Note:** XGBoost is not installed in this environment, so per the project's own fallback guidance, `GradientBoostingClassifier` from scikit-learn is used in its place — same gradient-boosted-tree family, same interpretability profile.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, classification_report)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def evaluate(model, Xte, yte):
    pred = model.predict(Xte)
    proba = model.predict_proba(Xte)[:, 1]
    return {
        'Accuracy': accuracy_score(yte, pred), 'Precision': precision_score(yte, pred),
        'Recall': recall_score(yte, pred), 'F1': f1_score(yte, pred),
        'ROC_AUC': roc_auc_score(yte, proba)
    }

results, fitted = {}, {}

lr = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
lr.fit(X_train_scaled, y_train)
fitted['Logistic Regression'] = lr
results['Logistic Regression'] = evaluate(lr, X_test_scaled, y_test)

dt = DecisionTreeClassifier(max_depth=8, min_samples_leaf=25, class_weight='balanced', random_state=42)
dt.fit(X_train, y_train)
fitted['Decision Tree'] = dt
results['Decision Tree'] = evaluate(dt, X_test, y_test)

rf = RandomForestClassifier(n_estimators=300, max_depth=None, min_samples_leaf=3,
                             class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
fitted['Random Forest'] = rf
results['Random Forest'] = evaluate(rf, X_test, y_test)

gb = GradientBoostingClassifier(n_estimators=250, max_depth=3, learning_rate=0.08, random_state=42)
gb.fit(X_train, y_train)
fitted['Gradient Boosting'] = gb
results['Gradient Boosting'] = evaluate(gb, X_test, y_test)

comparison = pd.DataFrame(results).T[['Accuracy','Precision','Recall','F1','ROC_AUC']].round(4)
comparison.sort_values('ROC_AUC', ascending=False)

## 8. Hyperparameter Tuning

A small manual comparison of Random Forest configurations, evaluated on a validation slice carved out of the training set (kept lightweight for this environment rather than a full grid search over CV folds).

In [ ]:
X_tr2, X_val, y_tr2, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)
configs = [
    {'n_estimators': 200, 'max_depth': 12, 'min_samples_leaf': 5},
    {'n_estimators': 300, 'max_depth': 14, 'min_samples_leaf': 4},
    {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 3},
]
for cfg in configs:
    m = RandomForestClassifier(**cfg, class_weight='balanced', random_state=42, n_jobs=-1)
    m.fit(X_tr2, y_tr2)
    auc = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
    print(cfg, '-> val ROC-AUC', round(auc, 4))

## 9. Model Evaluation

Why **not** just look at accuracy: with ~21% positive class, a model that never predicts churn scores ~78.6% accuracy while catching zero at-risk customers — worthless for retention. **Recall** (are we catching the customers who actually churn?) and **ROC-AUC** (how well does the model rank risk across the whole population?) matter more here, because the business cost of missing a churner (lost multi-year revenue) typically exceeds the cost of an unnecessary retention call to a customer who was never going to leave.

In [ ]:
best_model_name = comparison['ROC_AUC'].idxmax()
best_model = fitted[best_model_name]
print(f"Selected model: {best_model_name}")

X_eval = X_test_scaled if best_model_name == 'Logistic Regression' else X_test
y_pred = best_model.predict(X_eval)
y_proba = best_model.predict_proba(X_eval)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Active','Churned']))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=ax,
            xticklabels=['Active','Churned'], yticklabels=['Active','Churned'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {best_model_name}')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
fig, ax = plt.subplots(figsize=(6,5))
ax.plot(fpr, tpr, color='#e60000', label=f'{best_model_name} (AUC={roc_auc_score(y_test,y_proba):.3f})')
ax.plot([0,1],[0,1], linestyle='--', color='gray', label='Random guess')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve'); ax.legend()
plt.show()

## 10. Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X.columns)
else:
    importances = pd.Series(np.abs(best_model.coef_[0]), index=X.columns)

top20 = importances.sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(9,8))
top20.sort_values().plot(kind='barh', ax=ax, color='#e60000')
ax.set_title(f'Top 20 Features — {best_model_name}')
plt.tight_layout()
plt.show()
top20

## 11. Churn Probability & Risk Classification

In [ ]:
def risk_category(p):
    if p < 0.30: return 'Low'
    elif p < 0.60: return 'Medium'
    elif p < 0.80: return 'High'
    return 'Critical'

test_results = pd.DataFrame({
    'Actual_Churn': y_test.values,
    'Churn_Probability': y_proba,
})
test_results['Risk_Category'] = test_results['Churn_Probability'].apply(risk_category)
test_results['Risk_Category'].value_counts()

## 12. Risk Classification — validation against actual outcomes

In [ ]:
risk_validation = test_results.groupby('Risk_Category')['Actual_Churn'].agg(['count','mean'])
risk_validation.columns = ['Customers','Actual_Churn_Rate']
risk_validation['Actual_Churn_Rate'] = (risk_validation['Actual_Churn_Rate']*100).round(1)
risk_validation.loc[['Low','Medium','High','Critical']]

**Sanity check:** the actual churn rate observed within each predicted risk band should increase monotonically from Low to Critical — that's the real test of whether the risk tiers are meaningful, not just the aggregate ROC-AUC.

## 13. Final Model & Business Interpretation

In [ ]:
print("Final model: {}".format(best_model_name))
print(comparison.loc[best_model_name])
print()
print("Business interpretation:")
print("- The top features are dominated by service-quality and experience signals")
print("  (downtime, satisfaction, complaints, SLA breaches, uptime, NPS) rather than")
print("  commercial size/value fields — meaning churn here is genuinely a service")
print("  problem the network/support teams can act on, not just a pricing problem.")
print("- Recall of {:.1%} means the model catches a majority of actual churners,".format(results[best_model_name]['Recall']))
print("  at the cost of some false positives (unnecessary retention outreach) — an")
print("  acceptable trade given the asymmetric cost of losing an enterprise account.")

## 14. Export Model for Production Use

Full scoring, risk tiering, CLV, retention priority scoring, and per-customer explanations are handled by `src/prediction.py`, which is what powers the Streamlit app and the `churn_predictions` SQL table. See that file for the production-ready version of this pipeline.

In [ ]:
import joblib
artifact = {
    'model': best_model,
    'model_name': best_model_name,
    'scaler': scaler if best_model_name == 'Logistic Regression' else None,
    'feature_names': X.columns.tolist(),
    'encoder_categories': encoder_categories,
    'uses_scaling': best_model_name == 'Logistic Regression',
}
joblib.dump(artifact, '../models/churn_model.pkl')
print("Model saved to ../models/churn_model.pkl")